# 02 Moving Average Strategy

第二课目标：把买入持有变成一个最简单的交易策略。核心是理解 `signal`、`position` 和 `shift(1)`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.market_data import download_ohlcv
from quant_trading.moving_average import (
    add_moving_average_strategy,
    format_moving_average_summary,
    plot_moving_average_strategy,
    summarize_moving_average_strategy,
)

## 1. 策略规则

- 20 日均线 > 100 日均线：产生持有信号
- 20 日均线 <= 100 日均线：产生空仓信号
- 今天收盘后才知道今天的信号，所以明天才执行
- 单边交易成本暂设为 1 bps，也就是 0.01%

In [ ]:
SYMBOL = "SPY"
START = "2000-01-01"
SHORT_WINDOW = 20
LONG_WINDOW = 100
TRANSACTION_COST_BPS = 1.0

df = download_ohlcv(symbol=SYMBOL, start=START, auto_adjust=True)
strategy = add_moving_average_strategy(
    df,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

strategy.head()

## 2. 看懂核心字段

- `ma_short`：短期均线
- `ma_long`：长期均线
- `signal`：今天收盘后产生的信号
- `position`：当天实际持仓
- `trade`：当天是否发生买入或卖出
- `strategy_return`：策略当天收益率
- `strategy_equity`：策略净值曲线

In [ ]:
strategy[
    [
        "Close",
        "return",
        "ma_short",
        "ma_long",
        "signal",
        "position",
        "trade",
        "strategy_return",
        "strategy_equity",
        "strategy_drawdown",
    ]
].tail(10)

## 3. 为什么要 `shift(1)`

`signal` 是今天收盘后才知道的结果，不能用来赚今天的钱。所以策略实际持仓必须是昨天信号决定的：

```python
position = signal.shift(1)
```

In [ ]:
strategy[["Close", "ma_short", "ma_long", "signal", "position"]].dropna().head(8)

## 4. 比较买入持有和双均线策略

In [ ]:
summary = summarize_moving_average_strategy(
    strategy,
    symbol=SYMBOL,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
print(format_moving_average_summary(summary))

## 5. 画图

重点观察第二张图：策略净值曲线和买入持有净值曲线已经不一样了。

In [ ]:
fig = plot_moving_average_strategy(
    strategy,
    symbol=SYMBOL,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
)
fig